## Notebook 00: Database Setup

This notebook sets up the Brightway2 projects and imports the required life cycle inventory databases.

**What it does:**
1. Creates Brightway projects (`sense_project`, `sense_validation_geometric`, `sense_validation_arithmetic`).
2. Imports ecoinvent 3.10.1 (cutoff) and ecoinvent 3.11 (cutoff) via `bw2io.import_ecoinvent_release`.
3. Installs regioinvent on top of the ecoinvent 3.10.1 project using trade data from Zenodo.

**Inputs:**
- Ecoinvent credentials (username/password for ecoQuery).
- Regioinvent trade database (download from Zenodo record 15474318).

**Outputs:**
- Brightway projects with installed ecoinvent and regioinvent databases, ready for use in subsequent notebooks.

In [ ]:
# Enable automatic reloading of modified modules
%load_ext autoreload
%autoreload 2

import bw2data as bd
import bw2io as bi
import os
from pathlib import Path

In [ ]:
def get_ecoinvent_credentials():
    """Get ecoinvent credentials from environment variables or manual specification."""
    # Try environment variables first
    username = "username"
    password = "password"
    
    # Fallback to manual specification (uncomment lines above if needed)
    if not username or not password:
        try:
            return globals()['username'], globals()['password']
        except KeyError:
            print("⚠️  Please uncomment and set username/password variables above")
            return None, None
    
    return username, password

### Ecoinvent setup

In [ ]:
PROJECT = "sense_project"
EI_NAME = "ecoinvent-3.10.1-cutoff"

#username = "your_username"  # Replace with actual username
#password = "your_password"  # Replace with actual password

# 1) Ensure project exists / is selected
bd.projects.set_current(PROJECT)

# 2) Import ecoinvent 3.10 cutoff if missing
if EI_NAME in bd.databases:
    print(f"Database '{EI_NAME}' already exists in project '{bd.projects.current}'.")
else:
    username, password = get_ecoinvent_credentials()
    bi.import_ecoinvent_release(
        version="3.10.1",
        system_model="cutoff",
        username=username,
        password=password,
        use_mp=True,
    )
    print(f"Database '{EI_NAME}' installed successfully.")

### Validation Setup (for soca)

In [ ]:
PROJECT = "sense_validation_geometric"
EI_NAME = "ecoinvent-3.11-cutoff"

# 1) Ensure project exists / is selected
bd.projects.set_current(PROJECT)

# 2) Import ecoinvent 3.11 cutoff if missing
if EI_NAME in bd.databases:
    print(f"Database '{EI_NAME}' already exists in project '{bd.projects.current}'.")
else:
    username, password = get_ecoinvent_credentials()
    bi.import_ecoinvent_release(
        version="3.11",
        system_model="cutoff",
        username=username,
        password=password,
        use_mp=True,
    )
    print(f"Database '{EI_NAME}' installed successfully.")

In [ ]:
PROJECT = "sense_validation_arithmetic"
EI_NAME = "ecoinvent-3.11-cutoff"

# 1) Ensure project exists / is selected
bd.projects.set_current(PROJECT)

# 2) Import ecoinvent 3.11 cutoff if missing
if EI_NAME in bd.databases:
    print(f"Database '{EI_NAME}' already exists in project '{bd.projects.current}'.")
else:
    username, password = get_ecoinvent_credentials()
    bi.import_ecoinvent_release(
        version="3.11",
        system_model="cutoff",
        username=username,
        password=password,
        use_mp=True,
    )
    print(f"Database '{EI_NAME}' installed successfully.")

### Regioinvent

In [ ]:
import regioinvent

# --- CONFIGURATION ---
# Update these paths and names for your local setup
BW_PROJECT = "sense_project"
EI_DB_NAME = "ecoinvent-3.10.1-cutoff"
EI_VERSION = '3.10.1'  # Options: '3.9', '3.9.1', '3.10', '3.10.1'

# Path to the .db file downloaded from Zenodo
TRADE_DB_PATH = Path("../Data/regioinvent/trade_data.db")
OUTPUT_DB_NAME = 'Regioinvent'
TRADE_CUTOFF = 0.99  # 0 to 1; higher values take longer (2-3 hours)

In [ ]:
bd.projects.set_current(BW_PROJECT) # Use your project name
print(bd.databases)

In [ ]:
# 1. Initialize the Regioinvent class
print(f"Initializing Regioinvent for project: {BW_PROJECT}...")
regio = regioinvent.Regioinvent(
    bw_project_name=BW_PROJECT, 
    ecoinvent_database_name=EI_DB_NAME, 
    ecoinvent_version=EI_VERSION
)

In [ ]:
# 2. Spatialize the elementary flows
# Creates 'biosphere3_spatialized_flows' and a regionalized copy of ecoinvent
print("Step 1: Spatializing ecoinvent database...")
regio.spatialize_my_ecoinvent()

Patch impact methods (if necessary)

In [ ]:
import bw2data as bd
from collections import Counter, defaultdict
import pandas as pd

def patch_methods_and_print_replaced_biospheres(
    *,
    standard_biosphere_name: str,
    allowed_flow_dbs: set[str],
    dry_run: bool = True,
    validate_standard_flow_exists: bool = True,
    patch_only_if_db_name_contains: tuple[str, ...] | None = ("biosphere",),
    method_family_contains: str | None = None,
    max_methods_print: int = 40,
) -> pd.DataFrame:

    if standard_biosphere_name not in bd.databases:
        raise ValueError(f"Standard biosphere '{standard_biosphere_name}' not found in this project.")

    allowed_flow_dbs = set(allowed_flow_dbs)
    allowed_flow_dbs.add(standard_biosphere_name)

    standard_db = bd.Database(standard_biosphere_name)

    def standard_flow_exists(code: str) -> bool:
        if not validate_standard_flow_exists:
            return True
        try:
            standard_db.get(code)
            return True
        except Exception:
            return False

    scanned = 0
    candidate_methods = 0
    methods_with_rewrites = 0

    # diagnostics containers
    per_method_replaced = defaultdict(Counter)   # method -> Counter(src_db -> count)
    global_replaced = Counter()                  # src_db -> total rows replaced
    per_method_skipped_missing = Counter()       # method -> skipped rows

    rows = []

    for m in list(bd.methods):
        if method_family_contains and method_family_contains.lower() not in m[0].lower():
            continue

        data = bd.Method(m).load()
        if not data:
            continue

        scanned += 1
        referenced = set(r[0][0] for r in data)
        disallowed = referenced - allowed_flow_dbs

        if patch_only_if_db_name_contains is not None:
            disallowed = {
                dbn for dbn in disallowed
                if any(k.lower() in dbn.lower() for k in patch_only_if_db_name_contains)
            }

        if not disallowed:
            continue

        candidate_methods += 1

        new_data = []
        rewritten = 0
        skipped = 0

        for row in data:
            fk, cf, *rest = row
            dbn, code = fk

            if dbn in disallowed:
                if standard_flow_exists(code):
                    fk = (standard_biosphere_name, code)
                    rewritten += 1
                    per_method_replaced[str(m)][dbn] += 1
                    global_replaced[dbn] += 1
                else:
                    skipped += 1

            new_data.append((fk, cf, *rest))

        if rewritten > 0:
            methods_with_rewrites += 1
            if not dry_run:
                bd.Method(m).write(new_data)

        per_method_skipped_missing[str(m)] += skipped

        rows.append({
            "method": str(m),
            "rows_total": len(data),
            "disallowed_db_refs": "; ".join(sorted(disallowed)),
            "rows_rewritten": rewritten,
            "rows_skipped_missing_standard_flow": skipped,
            "replaced_from": "; ".join(
                f"{src} ({cnt})" for src, cnt in per_method_replaced[str(m)].most_common()
            ),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        print("No candidate methods found (no disallowed flow DB refs under current settings).")
        return df
    df = df.sort_values(["rows_rewritten", "rows_total"], ascending=False)

    # ---- PRINT DIAGNOSTICS ----
    print(f"Standard biosphere: {standard_biosphere_name}")
    print(f"Allowed flow DBs: {sorted(allowed_flow_dbs)}")
    print(f"Dry run: {dry_run}")
    print(f"Scanned methods: {scanned}")
    print(f"Candidate methods (had disallowed refs): {candidate_methods}")
    print(f"Methods with rewrites: {methods_with_rewrites}")
    print()

    if global_replaced:
        print("Biosphere DBs replaced (global totals):")
        for src, cnt in global_replaced.most_common():
            print(f"  {src}  ->  {standard_biosphere_name}   ({cnt} CF rows)")
    else:
        print("No replacements performed / possible under current settings.")
        print("If you expected replacements, the most common reason is UUIDs missing in the standard biosphere")
        print("while validate_standard_flow_exists=True.")
    print()

    if not df.empty:
        to_show = df[df["rows_rewritten"] > 0].head(max_methods_print)
        if to_show.empty:
            # show top candidates anyway
            to_show = df.head(max_methods_print)
            print(f"Top {len(to_show)} candidate methods (no rewrites):")
        else:
            print(f"Top {len(to_show)} methods with rewrites:")

        print(to_show[[
            "method",
            "rows_rewritten",
            "rows_skipped_missing_standard_flow",
            "replaced_from",
            "disallowed_db_refs"
        ]].to_string(index=False))
    else:
        print("No candidate methods found.")

    return df


In [ ]:
# 3. Import LCIA methods compatible with spatialized flows
print("Step 3: Importing regionalized LCIA methods...")
regio.import_fully_regionalized_impact_method(lcia_method='all')

print("Step 3b: Harmonizing biosphere references inside regionalized methods...")
allowed = {
    "biosphere3_spatialized_flows",
    "sociosphere",
    "ecoinvent-3.10.1-biosphere",
}

df = patch_methods_and_print_replaced_biospheres(
    standard_biosphere_name="ecoinvent-3.10.1-biosphere",
    allowed_flow_dbs=allowed,
    dry_run=False,   # start here
    max_methods_print=5,
)


In [ ]:
# 4. Regionalize with trade data
# This step is the long one (2-3 hours depending on cutoff)
print(f"Step 3: Regionalizing with trade data (Cutoff: {TRADE_CUTOFF})...")
regio.regionalize_ecoinvent_with_trade(
    trade_database_path=TRADE_DB_PATH,
    regioinvent_database_name=OUTPUT_DB_NAME,
    cutoff=TRADE_CUTOFF
)